In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import logging
logging.basicConfig(level=logging.INFO)

## Simulating ZTF SN Ia
This notebook simulates SN Ia light curves with realistic cadence, rate, parameter distributions, etc

In [3]:
quick_test = False
NSIM = 3000
RANDSEED = 1011

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Circle
from scipy.interpolate import interp1d
import pandas as pd
import sncosmo
from scipy import stats
from scipy.special import expit
from nested_pandas import read_parquet
from joblib import Parallel, delayed
import cloudpickle as pickle
from regions import RectangleSkyRegion
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.time import Time
import inspect
from IPython.display import display, Markdown

from lightcurvelynx.obstable.ztf_obstable import ZTFObsTable
from lightcurvelynx.astro_utils.passbands import PassbandGroup
from lightcurvelynx.astro_utils.pzflow_node import PZFlowNode
from lightcurvelynx.astro_utils.snia_utils import (
    DistModFromRedshift,
    HostmassX1Func,
    X0FromDistMod,
    num_snia_per_redshift_bin,
    SNCoordGivenPhysicalSep,
    snia_volumetric_rates,
)
from lightcurvelynx.math_nodes.scipy_random import SamplePDF
from lightcurvelynx.math_nodes.np_random import NumpyRandomFunc
from lightcurvelynx.simulate import simulate_lightcurves
from lightcurvelynx.models.sncosmo_models import SncosmoWrapperModel
from lightcurvelynx.models.snia_host import SNIaHost
from lightcurvelynx.utils.plotting import plot_lightcurves
from lightcurvelynx.math_nodes.ra_dec_sampler import ObsTableUniformRADECSampler
from lightcurvelynx.astro_utils.dustmap import DustmapWrapper,SFDMap
from lightcurvelynx.effects.extinction import ExtinctionEffect
from lightcurvelynx.astro_utils.mag_flux import mag2flux,flux2mag
from lightcurvelynx.astro_utils.detector_footprint import DetectorFootprint
from lightcurvelynx.utils.extrapolate import LinearDecayOnMag,ZeroPadding

from lightcurvelynx import _LIGHTCURVELYNX_BASE_DATA_DIR

from utils.lcfit import fit_single_lc
from utils.plotting_utils import plot_coverage_map
from utils.analysis_utils import lc_quality_cuts, spec_selection_func

from ztf_snia_sim_params import SIM_PARAMS

INFO:numexpr.utils:NumExpr defaulting to 10 threads.


In [5]:
RNG = np.random.default_rng(RANDSEED)

In [6]:
# load and define constants that are used throughout the simulations
H0 = SIM_PARAMS["H0"]
OMEGA_M = SIM_PARAMS["Omega_m"]
ALPHA = SIM_PARAMS["alpha"]
BETA = SIM_PARAMS["beta"]
ZMIN = SIM_PARAMS["zmin"]
ZMAX = SIM_PARAMS["zmax"]
MAGABS = SIM_PARAMS["mag_abs"]
SIGMA_MAGABS = SIM_PARAMS["sigma_mag_abs"]

ZP_ERR_MAG = SIM_PARAMS["zp_mag_err"]

In [7]:
globalhostdata = pd.read_csv('ztfsniadr2/tables/globalhost_data.csv')
localhostdata = pd.read_csv('ztfsniadr2/tables/localhost_data.csv')
sndata = pd.read_csv('ztfsniadr2/tables/snia_data.csv')
data = pd.merge(sndata,globalhostdata,on='ztfname')

In [8]:
lcdata = read_parquet('data/ztfsniadr2.parquet')

In [9]:
%%time

obs_log = pd.read_parquet('data/ztf_observing_log_combined_w_metadata.parquet')
colmap = {"ra":"ra",
          "dec":"dec",
          "time":"mjd",
          "zp":"zp_nJy",
          "filter":"filter",
          "sky":"sky_adu_ztfsn",
         }

#ztf ccd size 6144 × 6160 pixel * 16
pixel_scale = 1.01 #arcsec/pixel
center = SkyCoord(ra=0.0, dec=0.0, unit="deg", frame="icrs")
rect_region = RectangleSkyRegion(center=center, width=4*6144.* pixel_scale * u.arcsec, 
                                 height=4*6160.* pixel_scale * u.arcsec, angle=0.0 * u.deg)
ztf_fp = DetectorFootprint(rect_region, pixel_scale=pixel_scale)

ztf_obstable = ZTFObsTable(obs_log,colmap=colmap,detector_footprint=ztf_fp)
ztf_obstable.survey_values["zp_err_mag"] = ZP_ERR_MAG

t_min, t_max = ztf_obstable.time_bounds()
print(f"Loaded OpSim with {len(ztf_obstable)} rows and times [{t_min}, {t_max}]")

# #this is slow, so we only do it once, the total ZTF sky coverage given by the obstable is 
# ztf_obstable._table = ztf_obstable._table.drop_duplicates(subset=['ra','dec'])
# sky_coverage = ztf_obstable.estimate_coverage(use_footprint=True)
# print(f"The total sky coverage is {sky_coverage} square degrees")
# sky_coverage = 31981 #deg^2

passband_group = PassbandGroup.from_preset(preset="ZTF", filters=["g", "r", "i"])
print(f"Loaded Passbands: {passband_group}")

INFO:lightcurvelynx.obstable.obs_table:Provided radius 3.868 is smaller than footprint radius 4.871029037867509. Using the footprint radius instead.
INFO:lightcurvelynx.astro_utils.passbands:Loading passbands from preset ZTF


Loaded OpSim with 522192 rows and times [58288.171875, 59273.55859375]
Loaded Passbands: PassbandGroup containing 3 passbands: ZTF_g, ZTF_r, ZTF_i
CPU times: user 519 ms, sys: 97.1 ms, total: 616 ms
Wall time: 546 ms


In [10]:
t = Time([t_min,t_max], format='mjd', scale='utc')
t.to_datetime()

array([datetime.datetime(2018, 6, 19, 4, 7, 30),
       datetime.datetime(2021, 2, 28, 13, 24, 22, 500000)], dtype=object)

In [11]:
ztf_obstable._table.columns

Index(['time', 'band', 'fieldid', 'fieldra', 'fielddec', 'rcid', 'maglimit',
       'zp_abmag', 'gain', 'expid', 'infobits', 'skynoise', 'filter',
       'exptime', 'fwhm', 'obsdate', 'scibckgnd', 'ra', 'dec', 'maglim',
       'airmass', 'zp', 'sky', 'sky_adu_ztfmeta', 'obsmjd'],
      dtype='object')

In [12]:
# Load the Flow model into a PZFlow node. This gives access to all of the outputs of the
# flow model as attributes of the PZFlowNode.
pz_node = PZFlowNode.from_file("data/ztfsniadr2_host_sn_before_selection_pzflow.pkl",  # filename
    node_label="pznode",
)
radec_node = ObsTableUniformRADECSampler(ztf_obstable, node_label="radec")

# Create a model for the host of the SNIa. The attributes will be sampled via
# the PZFlowNode's model. So each hos instantiation will have its own properties.
# Note: This requires the user to know the output names from the underlying flow model.

rate = lambda z: 2.35e-5 #(2.35 ± 0.24) × 10^4 Gpc^−3 yr^−1 = (2.35 ± 0.24) × 10^4 * 10^-9 Mpc^−3 yr^−1, from ZTF BTS paper Perley et al. 2020

nsn, z = num_snia_per_redshift_bin(ZMIN, ZMAX, 100, H0=H0, Omega_m=OMEGA_M, vol_rate_function=rate)
zpdf = interp1d(z, nsn, bounds_error=False, fill_value=0)

host = SNIaHost(
    ra = radec_node.ra,
    dec = radec_node.dec,
    hostmass=pz_node.mass,
    redshift=SamplePDF(zpdf),
    node_label="host",
)

INFO:2026-03-05 11:58:13,804:jax._src.xla_bridge:812: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/Users/mi/anaconda3/envs/lightcurvelynx/bin/../lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file)
INFO:jax._src.xla_bridge:Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/Users/mi/anaconda3/envs/lightcurvelynx/bin/../lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file)


In [13]:
distmod_func = DistModFromRedshift(host.redshift, H0=H0, Omega_m=OMEGA_M)

m_abs_func = NumpyRandomFunc("normal", loc=MAGABS, scale=SIGMA_MAGABS)

# we model host-sn separation as an exponential distribution based on Fig 3 of Gupta et al 2016, mean separation = 5kpc
physical_host_sn_sep = NumpyRandomFunc("exponential", scale = 5.)
sncoor_node = SNCoordGivenPhysicalSep(host.ra, host.dec, physical_host_sn_sep, host.redshift, H0=H0, Omega_m=OMEGA_M,node_label='sncoor_node')

x0_func = X0FromDistMod(
    distmod=distmod_func,
    x1=pz_node.x1,
    c=pz_node.c,
    alpha=ALPHA,
    beta=BETA,
    m_abs=m_abs_func,
    node_label="x0_func",
)

sncosmo_modelname = "salt3"
time_extrap_before = ZeroPadding()
time_extrap_after = LinearDecayOnMag(decay_rate=0.02, mag_thres=30.)
wave_extrap_before = ZeroPadding()
wave_extrap_after = ZeroPadding()
source = SncosmoWrapperModel(
    sncosmo_modelname,
    t0=NumpyRandomFunc("uniform", low=t_min, high=t_max),
    x0=x0_func,
    x1=pz_node.x1,
    c=pz_node.c,
    ra=sncoor_node.ra,
    dec=sncoor_node.dec,
    redshift=host.redshift,
    node_label="source",
    time_extrapolation=(time_extrap_before,time_extrap_after),
    wave_extrapolation=(wave_extrap_before,wave_extrap_after),    
)
    
mwextinction = SFDMap(
    ra=source.ra,
    dec=source.dec,
    node_label="mwext",
)

# Create an extinction effect using the EBVs from that dust map.
ext_effect = ExtinctionEffect(extinction_model="F99", ebv=mwextinction, 
                              r_v=3.1,frame='observer',backend="dust_extinction")
source.add_effect(ext_effect)


INFO:lightcurvelynx.astro_utils.dustmap:SFD dust map data files not found.
Attempting to download from: ('https://github.com/kbarbary/sfddata/archive/master.tar.gz',)
to the directory /Users/mi/Work/lightcurvelynx/lightcurvelynx/data/dustmaps/sfdmap2


In [14]:
# save model and passbands to pickles
data_to_save = (source, passband_group)
with open("results/saved_model_and_passband.pkl", "wb") as file:
    pickle.dump(data_to_save, file)

In [15]:
### Notes:
### ZTF SN Ia DR2 dates: March(April 1) 2018 - December (Dec 31) 2020
### ZTF has square field?
### Data release has 3628 SN Ia in total
### 2667 passed data quality cut
### volume-limited complete sample to z<0.06

In [16]:
%%time
if quick_test:
    nsntotal = NSIM
else:
    survey_length = (t_max - t_min)/365.
    nsntotal, _ = num_snia_per_redshift_bin(ZMIN,ZMAX,1,solid_angle=9.136*survey_length,vol_rate_function=rate)
    print(f"Survey length = {survey_length} years")
print(f"Simulating {int(nsntotal)} SN ...")

# lightcurves = simulate_lightcurves(source, int(nsntotal), ztf_obstable, passband_group, 
#                                   param_cols = ['source.x0','source.x1','source.c','host.hostmass'],
#                                   obstable_save_cols=["infobits","airmass","fwhm","fieldid"])

lightcurves = simulate_lightcurves(
    model=source,
    num_samples=int(nsntotal),
    obstable=ztf_obstable,
    passbands=passband_group,
    param_cols = ['source.x0','source.x1','source.c','host.hostmass','source.ra','source.dec','host.ra','host.dec'],
    obstable_save_cols=["infobits","airmass","fwhm","fieldid"],
    num_jobs=8,
    batch_size=5000,
    rng=RNG,
)

lightcurves

<timed exec>:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
<timed exec>:15: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
INFO:lightcurvelynx.simulate:Using num_jobs=8 and batch_size=5000 for parallel processing.


Survey length = 2.6996896404109587 years
Simulating 105272 SN ...


AssertionError: nsamples must be a positive integer.

In [17]:
lightcurves['params'][0].keys()

NameError: name 'lightcurves' is not defined

In [ ]:
lightcurves.lightcurve.isna().sum()

In [ ]:
plot_coverage_map(ztf_obstable,lightcurves,plot_na_location=True,plot_all_location=False)

In [ ]:
idx = lightcurves.lightcurve.isna()

plt.subplot(2,1,1)
ztf_obstable._table.ra.hist(bins=100,density=False)
lightcurves.ra.hist(bins=100,alpha=0.8,density=False)
lightcurves.loc[idx].ra.hist(bins=100)
plt.yscale('log')
plt.xlabel('ra')

plt.subplot(2,1,2)
ztf_obstable._table.dec.hist(bins=100,density=False)
lightcurves.dec.hist(bins=100,alpha=0.8,density=False)
lightcurves.loc[idx].dec.hist(bins=100)
plt.yscale('log')
plt.xlabel('dec')

plt.tight_layout()

In [ ]:
# calculate detection flag
lightcurves = lightcurves.dropna(subset=['lightcurve'])

print("Before applying detection: nsn=", len(lightcurves))
lightcurves['lightcurve.snr'] = lightcurves['lightcurve.flux']/lightcurves['lightcurve.fluxerr']
detection_snr_thres = 5.
lightcurves['lightcurve.detection_flag'] = lightcurves['lightcurve.snr'] > detection_snr_thres

# drop saturation
lightcurves_after_drop_sat = lightcurves.query("lightcurve.is_saturated==False").dropna(subset=['lightcurve'])
print("After droppoing saturation: nsn=", len(lightcurves_after_drop_sat))

lightcurves_after_detection = lightcurves_after_drop_sat.query("lightcurve.detection_flag == True").dropna(subset=['lightcurve'])
print("After applying detection: nsn=", len(lightcurves_after_detection))

ZTF Selection Function (Fig 4 of ZTF DR2 Overview paper https://arxiv.org/pdf/2409.04346)
<img src="figs/ztf_selection_function.png" width="800" height="400">
<img src="figs/ztf_selection_function_formulism.png" width="800" height="400">

In [ ]:
m = np.linspace(15,20,50)
m0=18.8
s=4.5
p = np.power(1. + np.exp((m - m0)*s), -1)
plt.plot(m,p)

In [ ]:
# The selection function is defined as below.
display(Markdown(f"```python\n{inspect.getsource(spec_selection_func)}\n```"))

In [ ]:
# apply the above selection function

pass_selection = lightcurves_after_detection.reduce(spec_selection_func,"lightcurve.flux")
idx = pass_selection.query("pass_spec_selection == True").index
lightcurves_after_spec_selection = lightcurves_after_detection.loc[idx]
print("After spectroscopic selection: nsn=", len(lightcurves_after_spec_selection))

lightcurves["pass_spec_selection"] = False
lightcurves.loc[idx,"pass_spec_selection"] = True

In [ ]:
## ZTF SN DR2 selection cuts
# Start 3795
# No ZTF light curve -17 = 3778
# No spectra -110 = 3668
# No confirmed Ia type from spectra -40 = 3628
# Good sampling (7 different phases, 2 before, 2 after peak, 2 bands) = 2960, peak based on a salt2 fit
# SALT2 cuts = 2667

#### ZTF SN DR2 selection cuts
<img src="figs/ztf_selection_Table1_Rigault2024.png" width="800" height="400">

#### Good sampling cuts
<img src="figs/good_sample_cut.png" width="600" height="200">
<img src="figs/good_sample_cut2.png" width="600" height="200">

In [ ]:
# The lc_quality_cuts is defined as below
display(Markdown(f"```python\n{inspect.getsource(lc_quality_cuts)}\n```"))

In [ ]:
pass_quality_cut = lightcurves_after_spec_selection.reduce(lc_quality_cuts,"lightcurve.flux",
                                                           "lightcurve.mjd","lightcurve.filter","z")
idx = pass_quality_cut.query("pass_quality_cuts == True").index
lightcurves_after_quality_cut = lightcurves_after_spec_selection.loc[idx]
print("After quality cuts: nsn=", len(lightcurves_after_quality_cut))

lightcurves["pass_quality_cuts"] = False
lightcurves.loc[idx,"pass_quality_cuts"] = True

In [ ]:
lightcurves.head()

In [ ]:
#saving to parquet file
lightcurves.to_parquet("results/lightcurves.parquet")

In [ ]:
def fit_single_lc_w_cond(lc,
                         bounds={"x1": (-5,5),
                                 "c": (-0.4,1),},
                         phase_range=(-10,40),
                         modelcov=False):
    return fit_single_lc(lc,mpbounds=bounds,phase_range=phase_range,modelcov=modelcov)

In [ ]:
res = fit_single_lc_w_cond(lightcurves_after_quality_cut.iloc[0])
res

In [ ]:
lc_to_fit = lightcurves_after_quality_cut.iloc[0:]

In [ ]:
# %%time
# result_df0 = lc_to_fit.apply(fit_single_lc,axis=1)

In [ ]:
%%time
results = Parallel(n_jobs=10)(delayed(fit_single_lc_w_cond)(row) for _index, row in lc_to_fit.iterrows())
result_df = pd.DataFrame(results)

In [ ]:
result_df = result_df.set_index('id')

In [ ]:
result_df.dropna(subset=['success'])

In [ ]:
result_df.to_csv('results/salt3fit_results.csv')